In [ ]:
import os
import sys
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
from matplotlib.ticker import LogLocator, LogFormatterMathtext, NullFormatter
from typing import Optional, Sequence, Union, Mapping


project_root = os.path.abspath("/home/jovyan/work")

if project_root not in sys.path:
    sys.path.append(project_root)

from funcs import core, viz

In [ ]:
def prepare_metric_df(metric_df: pd.DataFrame) -> tuple[np.ndarray, pd.DataFrame]:
    """
    metric_df: index=subject, columns=t（float or numeric str）
    return: (t_sorted, df_sorted_float)
    """
    t_vals = pd.to_numeric(metric_df.columns, errors="raise").to_numpy()
    if np.any(t_vals <= 0):
        bad = t_vals[t_vals <= 0]
        raise ValueError(f"t must be > 0 for log scale. Found: {bad}")

    df = metric_df.astype(float)

    return t_vals, df


def _apply_log_x_pretty(ax: plt.Axes, xlim: tuple[float, float]) -> None:
    ax.set_xscale("log")
    ax.set_xlim(*xlim)

    ax.xaxis.set_major_locator(LogLocator(base=10.0, numticks=6))
    ax.xaxis.set_major_formatter(LogFormatterMathtext(base=10.0))

    ax.xaxis.set_minor_locator(LogLocator(base=10.0, subs=np.arange(2, 10) * 0.1, numticks=100))
    ax.xaxis.set_minor_formatter(NullFormatter())

    ax.grid(True, which="major", linestyle="--", alpha=0.3)
    ax.grid(True, which="minor", linestyle=":", alpha=0.1)


In [ ]:
def plt_metric_individual(
    metric_df: pd.DataFrame,
    metric_name: str,
    output_dir: str,
    nrows: int = 5,
    ncols: int = 4,
    ylim: Optional[tuple[float, float]] = None,
    xlim: Optional[tuple[float, float]] = None,
    yticks_all: bool = False,
    xticks_all: bool = False,
) -> list[str]:

    os.makedirs(output_dir, exist_ok=True)
    t_sorted, df = prepare_metric_df(metric_df)

    subjects = list(df.index)
    subjects_per_page = nrows * ncols
    n_total = len(subjects)
    n_pages = int(np.ceil(n_total / subjects_per_page))

    # xlim / ylim
    x_lim = xlim if xlim is not None else (float(t_sorted.min()), float(t_sorted.max()))
    if ylim is not None:
        y_lim = ylim
    else:
        y_lim = None

    saved = []

    for page in range(n_pages):
        start = page * subjects_per_page
        end = min((page + 1) * subjects_per_page, n_total)
        sub_list = subjects[start:end]

        fig, axes = plt.subplots(
            nrows=nrows, ncols=ncols,
            figsize=(ncols * 3.2, nrows * 3.2),
            sharex=True, sharey=True
        )
        axes = np.array(axes).reshape(-1)

        for k, ax in enumerate(axes):
            if k >= len(sub_list):
                ax.axis("off")
                continue

            sub = sub_list[k]
            y = df.loc[sub].to_numpy()

            ax.plot(t_sorted, y)
            _apply_log_x_pretty(ax, x_lim)

            if y_lim is not None:
                ax.set_ylim(*y_lim)

            ax.set_title(f"Subject {sub}", fontsize=14)

        if yticks_all:
            for ax in axes:
                if not ax.axison:
                    continue
                ax.tick_params(axis="y", which="both", labelleft=True, labelsize=11)

        if xticks_all:
            for ax in axes:
                if not ax.axison:
                    continue
                ax.tick_params(axis="x", which="both", labelbottom=True, labelsize=11)

        fig.text(0.5, 0.04, "Ratio (log scale)", ha="center", fontsize=16)
        fig.text(0.04, 0.5, metric_name, va="center", rotation="vertical", fontsize=16)
        fig.subplots_adjust(
            left=0.08, right=0.99,
            bottom=0.10, top=0.94,
            wspace=0.25, hspace=0.55
            )

        out_png = os.path.join(output_dir, f"{metric_name}_page{page+1:02d}.png")
        fig.savefig(out_png, dpi=300, bbox_inches="tight")
        saved.append(out_png)
        plt.show()
        plt.close(fig)

    return saved

In [ ]:
SubjectID = Union[int, str]
def plt_metric_overlay(
    metric_df: pd.DataFrame,
    subjects: Sequence[SubjectID],
    metric_name: str,
    title: Optional[str] = None,
    show_aggregate: bool = True,
    aggregate: str = "median",   # "mean" or "median"
    show_band: bool = False,
    band_quantiles: tuple[float, float] = (0.25, 0.75),
    xlim: Optional[tuple[float, float]] = None,
    ylim: Optional[tuple[float, float]] = None,
    output_dir: Optional[str] = None,  # if None, figure won't be saved
    subject_labels: Optional[Mapping[SubjectID, str]] = None,
    alpha: Optional[float] = None,
    linewidth: float = 1.25,
) -> plt.Figure:

    t, df = prepare_metric_df(metric_df)
    x_lim = xlim if xlim is not None else (float(t.min()), float(t.max()))

    missing = [s for s in subjects if s not in df.index]
    if missing:
        raise KeyError(f"Subjects not found in metric_df.index: {missing}")

    if alpha is None:
        alpha = 0.25 if len(subjects) >= 10 else 0.7

    fig, ax = plt.subplots(figsize=(7.2, 4.6))

    Y = []
    for s in subjects:
        y = df.loc[s].to_numpy()
        Y.append(y)

        if len(subjects) <= 6:
            lab = subject_labels[s] if (subject_labels is not None and s in subject_labels) else f"Sub {s}"
        else:
            lab = "_nolegend_"

        ax.plot(t, y, alpha=alpha, linewidth=linewidth, label=lab)

    Y = np.vstack(Y)

    if show_band:
        qlo, qhi = band_quantiles
        lo = np.nanquantile(Y, qlo, axis=0)
        hi = np.nanquantile(Y, qhi, axis=0)
        ax.fill_between(t, lo, hi, alpha=0.15, label="_nolegend_")

    if show_aggregate:
        if aggregate == "mean":
            agg = np.nanmean(Y, axis=0)
            agg_label = "Mean"
        elif aggregate == "median":
            agg = np.nanmedian(Y, axis=0)
            agg_label = "Median"
        else:
            raise ValueError("aggregate must be 'mean' or 'median'")

        ax.plot(t, agg, linewidth=2.6, alpha=0.95, label=agg_label)

    _apply_log_x_pretty(ax, x_lim)

    ax.set_xlabel("Ratio (log scale)", fontsize=14)
    ax.set_ylabel(metric_name, fontsize=14)
    if title is not None:
        ax.set_title(title, fontsize=18)

    if ylim is not None:
        ax.set_ylim(*ylim)

    if len(subjects) <= 6:
        ax.legend(frameon=True, fontsize=14)
    else:
        if show_aggregate:
            ax.legend(frameon=True, fontsize=14)

    ax.tick_params(axis="x", which="both", labelsize=13)
    ax.tick_params(axis="y", which="both", labelsize=13)
    fig.tight_layout()

    if output_dir is not None:
        out_png = os.path.join(output_dir, f"{metric_name}_{title}.png")
        fig.savefig(out_png, dpi=300, bbox_inches="tight")
    plt.close(fig)

    return fig

In [ ]:
h5_path = project_root + "/data/processed/amy_processeddata_all.h5"
with pd.HDFStore(h5_path, mode="r") as store:
    genmag = store["/metrics/genmag"]
    spr = store["/metrics/spread"]

In [ ]:
output = os.path.join(project_root, "fig/individual")

viz.plt_metric_individual(
    genmag,
    metric_name="Generalized magnitude",
    output_dir=output,
    nrows=5, ncols=4,
    ylim=[-1, 24],
    xlim=None,
    xticks_all=True,
    yticks_all=True
)

viz.plt_metric_individual(
    spr,
    metric_name="Spread",
    output_dir=output,
    nrows=5, ncols=4,
    ylim=[-1, 24],
    xlim=None,
    xticks_all=True,
    yticks_all=True
)

In [ ]:
subs = list(range(10))
output = os.path.join(project_root, "fig/group")
viz.plt_metric_overlay(
    genmag,
    subs,
    "Generalized manigtude",
    "Sub 1-10",
    show_aggregate=True,
    aggregate="median",
    show_band=True,
    band_quantiles=(0.25, 0.75),
    xlim=None,
    ylim=[-1, 24],
    output_dir=output,
    subject_labels=None,
)

In [ ]:
subs = list(range(10))
output = os.path.join(project_root, "fig/group")
viz.plt_metric_overlay(
    spr,
    subs,
    "Spread",
    "Sub 1-10",
    show_aggregate=True,
    aggregate="median",
    show_band=True,
    band_quantiles=(0.25, 0.75),
    xlim=None,
    ylim=[-1, 24],
    output_dir=output,
    subject_labels=None,
)